# Linear Probing of MomentFM on Google Colab

In [1]:
!pip install --no-deps momentfm

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report

import numpy as np
from tslearn.datasets import UCR_UEA_datasets
import torch.nn.functional as F
from momentfm import MOMENTPipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [3]:
def load_lsst_data():
    """Load the LSST dataset and format dimensions."""
    # Load the LSST dataset from UEA archive
    ds = UCR_UEA_datasets()
    X_train, y_train, X_test, y_test = ds.load_dataset("LSST")
    
    # Swap axes to match (n_samples, n_channels, n_timesteps) format
    X_train = np.swapaxes(X_train, 1, 2)
    X_test = np.swapaxes(X_test, 1, 2)
    
    # Flatten labels for scikit-learn
    y_train = y_train.ravel()
    y_test = y_test.ravel()
    
    return X_train, y_train, X_test, y_test 

In [4]:
# Load LSST dataset
X_train, y_train, X_test, y_test = load_lsst_data()

# Encode labels to integers
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
num_classes = len(label_encoder.classes_)
print(num_classes)

# ---------------------------------------------------------
# STANDARD SCALING FOR 3D TIME SERIES
# ---------------------------------------------------------
samples_train, channels, timesteps = X_train.shape
samples_test = X_test.shape[0]

# Reshape from (samples, channels, timesteps) to (samples * timesteps, channels)
# This format allows the scaler to compute 1 mean and 1 std per channel
X_train_reshaped = X_train.transpose(0, 2, 1).reshape(-1, channels)
X_test_reshaped = X_test.transpose(0, 2, 1).reshape(-1, channels)

scaler = StandardScaler()

# Fit ONLY on the training data, then transform both sets
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_test_scaled = scaler.transform(X_test_reshaped)

# Reshape back to the original 3D shape (samples, channels, timesteps)
X_train = X_train_scaled.reshape(samples_train, timesteps, channels).transpose(0, 2, 1)
X_test = X_test_scaled.reshape(samples_test, timesteps, channels).transpose(0, 2, 1)
# ---------------------------------------------------------

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.long)


14


In [5]:
seq_len = 512
pad_len = seq_len - timesteps # 512 - 36 = 476

# Pad the time dimension (the last dimension) with zeros
X_train_padded = F.pad(X_train_tensor, (0, pad_len))
X_test_padded = F.pad(X_test_tensor, (0, pad_len))

# Create input masks (1 for real data, 0 for padding)
input_mask_train = torch.ones(samples_train, seq_len, dtype=torch.long)
input_mask_train[:, timesteps:] = 0

input_mask_test = torch.ones(samples_test, seq_len, dtype=torch.long)
input_mask_test[:, timesteps:] = 0

train_dataset = TensorDataset(X_train_padded, input_mask_train, y_train_tensor)
test_dataset = TensorDataset(X_test_padded, input_mask_test, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [6]:
model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={
        'task_name':'classification',
        'n_channels': 6,
        'num_class': 14
    }, # We are loading the model in `classification` mode
    # local_files_only=True,  # Whether or not to only look at local files (i.e., do not try to download the model).
)
model.init()
print(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


MOMENTPipeline(
  (normalizer): RevIN()
  (tokenizer): Patching()
  (patch_embedding): PatchEmbedding(
    (value_embedding): Linear(in_features=8, out_features=1024, bias=False)
    (position_embedding): PositionalEmbedding()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
  

/usr/local/lib/python3.12/dist-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


In [7]:
config = {
    "learning_rate": 1e-3,
    "epochs": 12,
    "batch_size": 32,
    "dataset": "LSST"
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = nn.CrossEntropyLoss()
# Pass only the newly initialized classifier parameters to the optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(config["epochs"]):
    # --- Training Phase ---
    model.train()
    total_loss = 0

    # Unpack the mask alongside X and y
    for batch_X, batch_mask, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_mask = batch_mask.to(device)
        batch_y = batch_y.to(device)

        # Pass the input mask to the model
        output = model(x_enc=batch_X, input_mask=batch_mask)

        # backward
        loss = criterion(output.logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # --- Evaluation Phase ---
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_X, batch_mask, batch_y in test_loader:
            batch_X = batch_X.to(device)
            batch_mask = batch_mask.to(device)
            
            output = model(x_enc=batch_X, input_mask=batch_mask)
            preds = torch.argmax(output.logits, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_targets.extend(batch_y.numpy())

    # Calculate robust metrics for imbalanced classes
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    bal_acc = balanced_accuracy_score(all_targets, all_preds)

    print(f"Epoch {epoch+1}/{config['epochs']} | Loss: {avg_train_loss:.4f} | Macro F1: {macro_f1:.4f} | Bal Acc: {bal_acc:.4f}")

# --- Final Classification Report ---
print("\n" + "="*50)
print("FINAL CLASSIFICATION REPORT (Test Set)")
print("="*50)
print(classification_report(all_targets, all_preds))


Epoch 1/12 | Loss: 2.2469 | Macro F1: 0.0342 | Bal Acc: 0.0714
Epoch 2/12 | Loss: 2.0227 | Macro F1: 0.0581 | Bal Acc: 0.0854
Epoch 3/12 | Loss: 1.9469 | Macro F1: 0.0913 | Bal Acc: 0.1198
Epoch 4/12 | Loss: 1.8829 | Macro F1: 0.0972 | Bal Acc: 0.1274


KeyboardInterrupt: 